In [0]:
# ! pip install holidays

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date
import holidays

In [0]:
from pyspark import SparkFiles
p1 ="https://github.com/erkansirin78/datasets/raw/master/Store_Item_Demand_Forecasting/train.csv"
sc.addFile(p1)
df = spark.read.csv("file:///"+SparkFiles.get("train.csv"), header=True, inferSchema=True).withColumnRenamed("sales", "label")

In [0]:
display(df.limit(5))

date,store,item,label
2013-01-01,1,1,13
2013-01-02,1,1,11
2013-01-03,1,1,14
2013-01-04,1,1,13
2013-01-05,1,1,10


In [0]:
display(df.groupBy("store").count())

store,count
1,91300
6,91300
3,91300
5,91300
9,91300
4,91300
8,91300
7,91300
10,91300
2,91300


In [0]:
display(df.groupBy("item").count())

item,count
12,18260
1,18260
13,18260
6,18260
3,18260
5,18260
9,18260
4,18260
8,18260
7,18260


In [0]:
df.printSchema()

root
 |-- date: date (nullable = true)
 |-- store: integer (nullable = true)
 |-- item: integer (nullable = true)
 |-- label: integer (nullable = true)



In [0]:
df_item_12 = df.filter("item = 12")

In [0]:
from pyspark.ml import Transformer
from pyspark.sql.functions import lit, udf
from pyspark.ml.param.shared import HasInputCols, HasOutputCol
from datetime import date

class AddDateFeaturesTransformer(Transformer, HasInputCols, HasOutputCol):
    def __init__(self, inputCol=None, outputCols=None, date=None, country_code=None):
        super(AddDateFeaturesTransformer, self).__init__()
        self.inputCol = inputCol
        self.outputCols = outputCols
        self.date = date
        self.country_code = country_code
    
    @udf(IntegerType())
    def is_holiday(date_str: date, country_code: str='TR'):
        date_str = str(self.date)
        country_holidays = holidays.CountryHoliday(country_code) 
        date_obj = date.fromisoformat(date_str)
        if date_obj in country_holidays:
            return 1
        else:
            return 0
    def _transform(self, df):
        df = df.withColumn(self.outputCols[0], F.year(self.inputCol)) \
        .withColumn(self.outputCols[1], F.month(self.inputCol)) \
        .withColumn(self.outputCols[2], F.dayofweek(self.inputCol)) \
        .withColumn(self.outputCols[3], is_holiday(self.inputCol))

        return df

In [0]:
sct = AddDateFeaturesTransformer(inputCol='date', outputCols=['y','m','d','is_holiday'])

In [0]:
from pyspark.ml.feature import OneHotEncoder

ohe = OneHotEncoder() \
.setInputCols(['item']) \
.setOutputCols(['item_ohe'])

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler().setHandleInvalid("skip") \
.setInputCols(['item_ohe','y','m','d','is_holiday']) \
.setOutputCol('features')

In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(numTrees=100)

In [0]:
from pyspark.ml.pipeline import Pipeline

pipeline_obj = Pipeline().setStages([ohe, sct, assembler, rf])

In [0]:
train_df, test_df = df_item_12.randomSplit([0.7, 0.3], seed=142)

In [0]:
pipeline_model = pipeline_obj.fit(train_df)

In [0]:
transformed_df = pipeline_model.transform(test_df)

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:
evaluator = RegressionEvaluator(metricName='r2')

In [0]:
r2 = evaluator.evaluate(transformed_df)

In [0]:
r2

Out[138]: 0.3246836532258687